# Hypothesis: Pre-Draft Background and Early NBA Performance

## Research Question

Does a player's pre-draft background influence his performance during the first three seasons in the NBA?

## Hypothesis

Players with stronger pre-draft characteristics tend to perform better during their first three NBA seasons.

In this project, pre-draft background includes draft position, draft round, physical measurements, and available combine statistics. Early NBA performance is measured using average player statistics during the first three seasons after entering the league.


## Key Variables

### Pre-Draft Characteristics

Possible explanatory variables:

* draft number
* height
* weight
* wingspan
* standing reach
* vertical leap
* position
* college or previous team

### Early NBA Performance

Performance during the first three NBA seasons can be measured using:

* average points per game
* average assists per game
* average rebounds per game
* average minutes played
* average games played in the season




In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy

In [10]:
draft_history_path = "..\data\processed\draft_history.csv"
draft_history = pd.read_csv(draft_history_path)
combine_draft_history_path = "..\data\processed\draft_combine_stats.csv"
combine_draft_history = pd.read_csv(combine_draft_history_path)

### Players

In [11]:
players = draft_history[draft_history["round_number"] == 1][["person_id", "player_name", "season", "overall_pick", "organization", "organization_type"]]
сombine_draft = combine_draft_history[['player_id','height_wo_shoes', 'weight', 'wingspan']]

In [12]:
players = players.rename(columns={'person_id': 'player_id'})
players = players.merge(
    сombine_draft,
    on="player_id",
    how="left"
)

In [13]:
players["weight"] = pd.to_numeric(players["weight"], errors="coerce")
players["imb"] = players["weight"] * 703 / players["height_wo_shoes"]**2


In [14]:
players

,player_id,player_name,season,overall_pick,organization,organization_type,height_wo_shoes,weight,wingspan,imb
0,79299,Clifton McNeeley,1947,1,Texas-El Paso,College/University,NaN,NaN,NaN,NaN
1,78109,Glen Selbo,1947,2,Wisconsin,College/University,NaN,NaN,NaN,NaN
2,76649,Eddie Ehlers,1947,3,Purdue,College/University,NaN,NaN,NaN,NaN
3,79302,Walt Dropo,1947,4,Connecticut,College/University,NaN,NaN,NaN,NaN
4,77048,Dick Holub,1947,5,Long Island-Brooklyn,College/University,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
1629,1641733,Nick Smith Jr.,2023,27,Arkansas,College/University,77.75,NaN,82.50,NaN
1630,1641729,Brice Sensabaugh,2023,28,Ohio State,College/University,77.75,NaN,82.50,NaN
1631,1631124,Julian Strawther,2023,29,Gonzaga,College/University,77.75,207.6,81.50,24.142480
1632,1631124,Julian Strawther,2023,29,Gonzaga,College/University,78.00,208.8,81.25,24.126627


### Work with game stats of players

In [15]:
pbp_path = "..\data\processed\play_by_play.csv"
pbp = pd.read_csv(pbp_path)

In [16]:
field_goals = pbp[pbp["eventmsgtype"] == 1]
free_throws = pbp[pbp["eventmsgtype"] == 3]
rebounds = pbp[pbp["eventmsgtype"] == 4]

In [17]:
field_goals["description"] = (
    field_goals["homedescription"]
    .replace("unknown", pd.NA)
    .fillna(field_goals["visitordescription"].replace("unknown", pd.NA))
)

free_throws["description"] = (
    free_throws["homedescription"]
    .replace("unknown", pd.NA)
    .fillna(free_throws["visitordescription"].replace("unknown", pd.NA))
)

rebounds["description"] = (
    rebounds["homedescription"]
    .replace("unknown", pd.NA)
    .fillna(rebounds["visitordescription"].replace("unknown", pd.NA))
)

In [18]:
field_goals["pts"] = 2
field_goals.loc[
    field_goals["description"].str.contains("3PT"),
    "pts"
] = 3

free_throws = free_throws[~free_throws['description'].str.contains('MISS')].copy()
free_throws["pts"] = 1



In [19]:
scoring_events = pd.concat([
    field_goals[["game_id", "player1_id", "player2_id", "pts"]],
    free_throws[["game_id", "player1_id", "pts"]]
], ignore_index=True).sort_values(by='game_id')

rebounds = rebounds[["game_id", 'player1_id']]


In [20]:
scoring_events



,game_id,player1_id,player2_id,pts
2938809,11300001,42545,NaN,1
2938810,11300001,200757,NaN,1
2938811,11300001,42544,NaN,1
2938812,11300001,42544,NaN,1
2938813,11300001,42544,NaN,1
...,...,...,...,...
2352559,49800087,764,NaN,1
2352556,49800087,990,NaN,1
2352557,49800087,1495,NaN,1
2352536,49800087,251,NaN,1


In [21]:
rebounds

,game_id,player1_id
3,29600012,406
5,29600012,208
9,29600012,1610612747
14,29600012,76
17,29600012,170
...,...,...
13585524,32200001,202681
13585527,32200001,1610616834
13585529,32200001,1628374
13585531,32200001,203944


In [41]:
player_game_points = scoring_events.groupby(["game_id", "player1_id"])["pts"].sum().reset_index()
player_total_points = player_game_points.groupby('player1_id').agg(
        total_pts=("pts", "sum"),
        total_games=("game_id", "nunique")
    ).reset_index().rename(columns={"player1_id": "player_id"})

player_total_points["pts_per_game"] = (
    player_total_points["total_pts"] / player_total_points["total_games"]
)

player_total_points

,player_id,total_pts,total_games,pts_per_game
0,0,2,1,2.000000
1,2,499,68,7.338235
2,3,1956,291,6.721649
3,7,548,91,6.021978
4,9,119,25,4.760000
...,...,...,...,...
2933,1962937772,7,1,7.000000
2934,1962937773,3,1,3.000000
2935,1962937808,9,1,9.000000
2936,1962937813,12,1,12.000000


In [39]:
player_total_assists = (
    scoring_events[scoring_events["player2_id"].notna() & (scoring_events["player2_id"] != 0)]
    .groupby("player2_id")
    .agg(
        total_assists=("game_id", "count"),
        total_games=("game_id", "nunique")
    )
    .reset_index()
    .rename(columns={"player2_id": "player_id"})
)

player_total_assists["assists_per_game"] = (
    player_total_assists["total_assists"] / player_total_assists["total_games"]
)

player_total_assists = player_total_assists[player_total_assists['player_id'] != 0]
player_total_assists

,player_id,total_assists,total_games,assists_per_game
0,2.000000e+00,96,55,1.745455
1,3.000000e+00,374,209,1.789474
2,7.000000e+00,63,43,1.465116
3,9.000000e+00,80,29,2.758621
4,1.200000e+01,1,1,1.000000
...,...,...,...,...
2773,1.962938e+09,1,1,1.000000
2774,1.962938e+09,2,1,2.000000
2775,1.962938e+09,3,1,3.000000
2776,1.962938e+09,1,1,1.000000


In [38]:
player_total_rebounds = (
    rebounds
    .groupby("player1_id")
    .agg(
        total_rebounds=("game_id", "count"),
        total_games=("game_id", "nunique")
    )
    .reset_index()
    .rename(columns={"player1_id": "player_id"})
)

player_total_rebounds["rebounds_per_game"] = (
    player_total_rebounds["total_rebounds"] / player_total_rebounds["total_games"]
)

player_total_rebounds

,player_id,total_rebounds,total_games,rebounds_per_game
0,0,1,1,1.000000
1,2,117,57,2.052632
2,3,1386,306,4.529412
3,6,6,1,6.000000
4,7,370,102,3.627451
...,...,...,...,...
3009,1962937772,2,1,2.000000
3010,1962937773,2,1,2.000000
3011,1962937808,4,1,4.000000
3012,1962937809,1,1,1.000000
